# Archivus – List Files: Sort & Filter

Exercises the sort and filter options added to `POST /storage/files`.

| Endpoint | Method |
|----------|--------|
| `/storage/files` | POST |

**Request body** (all fields JSON):
- `path` — folder relative to the drive root (`""` for root)
- `driveId` — the drive UUID
- `page`, `pageSize` — pagination (optional)
- `sortBy` — `"name"` (default), `"size"`, `"created_at"`
- `sortOrder` — `"asc"` (default) or `"desc"`
- `category` — content-type group: `images`, `videos`, `audio`, `spreadsheets`, `docs`, `pdfs`, `code`, `others`
- `contentType` — exact MIME type (used only when `category` is empty)

**Response** `files[]` entries include `CreatedAt` and `ContentType`.

**Pre-requisites:**
- Server running on `http://localhost:8080`
- An admin user must exist (cells below will create one if missing)

In [1]:
import requests, json, time

BASE   = 'http://localhost:8080'
# Unique folder per run so re-running the notebook never collides with
# fixtures (or stale metadata rows) left over from a previous run.
FOLDER = f'sort_filter_{int(time.time())}'

def show(resp):
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    print(f'Status : {resp.status_code}')
    print(f'Body   : {json.dumps(body, indent=2)}')
    return body

token    = None
drive_id = None

## 1 · Auth setup

Register + login as an admin user to get a token and the owner drive ID.  
If the user already exists the duplicate-register 400 is harmless.

In [2]:
requests.post(f'{BASE}/auth/register', json={
    'username': 'samar', 'password': 'password12', 'pin': '123456',
    'email': 'samar@example.com', 'user_type': 'business', 'is_admin': True,
})

resp = requests.post(f'{BASE}/auth/login', json={'username': 'samar', 'pin': '123456'})
assert resp.status_code == 200, f'login failed: {resp.status_code} {resp.text}'
token = resp.json()['token']

resp = requests.get(f'{BASE}/auth/user/info',
    headers={'Authorization': f'Bearer {token}'})
body = resp.json()
drives   = body.get('drives', [])
drive_id = drives[0]['DriveID'] if drives else None
print(f'drive_id : {drive_id}')
assert drive_id, 'drive_id is None — register/login failed'

drive_id : 191f0c71-b46a-4aa0-8d37-adedd3949974


## 2 · Create a folder and upload fixture files

Files span every category and have deliberately different sizes.  
A short sleep between uploads guarantees distinct `created_at` values  
so the `created_at` sort is deterministic.

In [3]:
# Best-effort reset: delete any leftover fixture folder from a previous run,
# so re-running the notebook starts from a clean slate.
requests.post(f'{BASE}/storage/folder/delete',
    headers={'Authorization': f'Bearer {token}'},
    json={'path': FOLDER, 'driveId': drive_id})

resp = requests.post(f'{BASE}/storage/folder/create',
    headers={'Authorization': f'Bearer {token}'},
    json={'path': FOLDER, 'driveId': drive_id})
assert resp.status_code == 200, f'create folder failed: {resp.status_code} {resp.text}'
print(f'✓ folder "{FOLDER}" created')

✓ folder "sort_filter_1787165718" created


In [4]:
# (name, bytes, content-type) — upload order == created_at order
fixtures = [
    ('first.txt',   b'x' * 10,                 'text/plain'),       # docs        (10 B)
    ('second.txt',  b'x' * 1000,               'text/plain'),       # docs      (1000 B)
    ('third.txt',   b'x' * 100,                'text/plain'),       # docs       (100 B)
    ('pic-a.png',   b'\x89PNG' + b'x' * 5000,  'image/png'),        # images    (5000 B)
    ('pic-b.png',   b'\x89PNG' + b'x' * 3000,  'image/png'),        # images    (3000 B)
    ('report.pdf',  b'%PDF' + b'x' * 2000,     'application/pdf'),  # pdfs      (2000 B)
    ('sheet.csv',   b'a,b,c\n' + b'x' * 500,   'text/csv'),         # spreadsheets (500 B)
    ('script.py',   b'print("hi")\n' + b'x' * 400, 'text/x-python'),# code      (400 B)
    ('song.mp3',    b'ID3' + b'x' * 6000,      'audio/mpeg'),       # audio     (6000 B)
    ('clip.mp4',    b'ftyp' + b'x' * 7000,     'video/mp4'),        # videos    (7000 B)
    ('archive.zip', b'PK\x03\x04' + b'x' * 1500, 'application/zip'), # others  (1500 B)
]

for name, data, ctype in fixtures:
    resp = requests.post(f'{BASE}/storage/file/upload',
        headers={'Authorization': f'Bearer {token}'},
        data={'folderPath': FOLDER, 'driveId': drive_id},
        files=[('files', (name, data, ctype))])
    assert resp.status_code == 200, f'upload {name} failed: {resp.status_code} {resp.text}'
    print(f'  uploaded {name:12s} ({len(data):5d} bytes, {ctype})')
    time.sleep(1.2)  # distinct created_at for the sort test

  uploaded first.txt    (   10 bytes, text/plain)
  uploaded second.txt   ( 1000 bytes, text/plain)
  uploaded third.txt    (  100 bytes, text/plain)
  uploaded pic-a.png    ( 5004 bytes, image/png)
  uploaded pic-b.png    ( 3004 bytes, image/png)
  uploaded report.pdf   ( 2004 bytes, application/pdf)
  uploaded sheet.csv    (  506 bytes, text/csv)
  uploaded script.py    (  412 bytes, text/x-python)
  uploaded song.mp3     ( 6003 bytes, audio/mpeg)
  uploaded clip.mp4     ( 7004 bytes, video/mp4)
  uploaded archive.zip  ( 1504 bytes, application/zip)


## 3 · Helpers

`list_files` POSTs to `/storage/files` and returns the parsed JSON body.  
`files_only` drops directory rows so the file-sort assertions are unaffected  
by the directories-first ordering.

In [5]:
def list_files(path, sortBy=None, sortOrder=None, category=None, contentType=None, page=1, pageSize=100):
    payload = {'path': path, 'driveId': drive_id, 'page': page, 'pageSize': pageSize}
    if sortBy is not None:
        payload['sortBy'] = sortBy
    if sortOrder is not None:
        payload['sortOrder'] = sortOrder
    if category is not None:
        payload['category'] = category
    if contentType is not None:
        payload['contentType'] = contentType
    resp = requests.post(f'{BASE}/storage/files',
        headers={'Authorization': f'Bearer {token}'},
        json=payload)
    assert resp.status_code == 200, f'list failed: {resp.status_code} {resp.text}'
    return resp.json()

def files_only(body):
    return [e for e in (body.get('files') or []) if not e.get('IsDir')]

def names(entries):
    return [e['Name'] for e in entries]

def sizes(entries):
    return [e['Size'] for e in entries]

## 4 · Baseline — default sort (name asc)

No `sortBy`/`sortOrder`/`category`/`contentType`: entries should come back name-ascending.

In [6]:
body = list_files(FOLDER)
entries = files_only(body)
got = names(entries)
print(f'Default listing names : {got}')
assert got == sorted(got), f'expected name-ascending order, got {got}'
assert body.get('total') == len(fixtures), f'expected total {len(fixtures)}, got {body.get("total")}'
print(f'✓ default order is name-ascending (total={len(fixtures)})')

Default listing names : ['archive.zip', 'clip.mp4', 'first.txt', 'pic-a.png', 'pic-b.png', 'report.pdf', 'script.py', 'second.txt', 'sheet.csv', 'song.mp3', 'third.txt']
✓ default order is name-ascending (total=11)


## 5 · Sort by `size`

`sortBy: "size"` with `sortOrder` `"asc"` then `"desc"`.  
Sizes are reported in MB, so compare relative ordering, not raw bytes.

In [7]:
body = list_files(FOLDER, sortBy='size', sortOrder='asc')
entries = files_only(body)
print('size asc :', list(zip(names(entries), sizes(entries))))
assert sizes(entries) == sorted(sizes(entries)), 'size ascending order violated'

body = list_files(FOLDER, sortBy='size', sortOrder='desc')
entries = files_only(body)
print('size desc:', list(zip(names(entries), sizes(entries))))
assert sizes(entries) == sorted(sizes(entries), reverse=True), 'size descending order violated'
print('✓ size sort (asc + desc) correct')

size asc : [('first.txt', 9.5367431640625e-06), ('third.txt', 9.5367431640625e-05), ('script.py', 0.000392913818359375), ('sheet.csv', 0.0004825592041015625), ('second.txt', 0.00095367431640625), ('archive.zip', 0.001434326171875), ('report.pdf', 0.001911163330078125), ('pic-b.png', 0.002864837646484375), ('pic-a.png', 0.004772186279296875), ('song.mp3', 0.005724906921386719), ('clip.mp4', 0.006679534912109375)]
size desc: [('clip.mp4', 0.006679534912109375), ('song.mp3', 0.005724906921386719), ('pic-a.png', 0.004772186279296875), ('pic-b.png', 0.002864837646484375), ('report.pdf', 0.001911163330078125), ('archive.zip', 0.001434326171875), ('second.txt', 0.00095367431640625), ('sheet.csv', 0.0004825592041015625), ('script.py', 0.000392913818359375), ('third.txt', 9.5367431640625e-05), ('first.txt', 9.5367431640625e-06)]
✓ size sort (asc + desc) correct


## 6 · Sort by `created_at`

Upload order is `first.txt → second.txt → … → archive.zip`,  
so ascending `created_at` must match that exact order (descending is the reverse).

In [8]:
upload_order = [n for n, _, _ in fixtures]

body = list_files(FOLDER, sortBy='created_at', sortOrder='asc')
entries = files_only(body)
print('created_at asc :', names(entries))
for e in entries:
    print(f'  {e["Name"]:12s} CreatedAt={e.get("CreatedAt")}')
assert names(entries) == upload_order, f'created_at asc expected {upload_order}'

body = list_files(FOLDER, sortBy='created_at', sortOrder='desc')
entries = files_only(body)
print('created_at desc:', names(entries))
assert names(entries) == list(reversed(upload_order)), 'created_at desc order violated'
print('✓ created_at sort (asc + desc) correct')

created_at asc : ['first.txt', 'second.txt', 'third.txt', 'pic-a.png', 'pic-b.png', 'report.pdf', 'sheet.csv', 'script.py', 'song.mp3', 'clip.mp4', 'archive.zip']
  first.txt    CreatedAt=2026-08-20T00:25:27.650076208+05:30
  second.txt   CreatedAt=2026-08-20T00:25:29.679540397+05:30
  third.txt    CreatedAt=2026-08-20T00:25:31.703209615+05:30
  pic-a.png    CreatedAt=2026-08-20T00:25:33.72627001+05:30
  pic-b.png    CreatedAt=2026-08-20T00:25:35.687107882+05:30
  report.pdf   CreatedAt=2026-08-20T00:25:37.543692561+05:30
  sheet.csv    CreatedAt=2026-08-20T00:25:39.384499569+05:30
  script.py    CreatedAt=2026-08-20T00:25:41.415757626+05:30
  song.mp3     CreatedAt=2026-08-20T00:25:43.456928784+05:30
  clip.mp4     CreatedAt=2026-08-20T00:25:45.305962676+05:30
  archive.zip  CreatedAt=2026-08-20T00:25:47.165826149+05:30
created_at desc: ['archive.zip', 'clip.mp4', 'song.mp3', 'script.py', 'sheet.csv', 'report.pdf', 'pic-b.png', 'pic-a.png', 'third.txt', 'second.txt', 'first.txt']
✓ cr

## 7 · Filter by `category`

Each category expands to a set of MIME types (`images`/`videos`/`audio` match by  
`image/`, `video/`, `audio/` prefix; the rest match an explicit list; `others` is  
the complement). `total` reflects the filtered file count.

In [11]:
expected = {
    'images':       ['pic-a.png', 'pic-b.png'],
    'videos':       ['clip.mp4'],
    'audio':        ['song.mp3'],
    'spreadsheets': ['sheet.csv'],
    'docs':         ['first.txt', 'second.txt', 'third.txt'],
    'pdfs':         ['report.pdf'],
    'code':         ['script.py'],
    'others':       ['archive.zip'],
}

for category, want in expected.items():
    body = list_files(FOLDER, category=category)
    entries = files_only(body)
    got = names(entries)
    print(f'{category:13s} -> {got}')
    assert got == want, f'category {category!r}: expected {want}, got {got}'
    assert body.get('total') == len(want), f'category {category!r}: expected total {len(want)}, got {body.get("total")}'
print('✓ all category filters correct')

images        -> ['pic-a.png', 'pic-b.png']
videos        -> ['clip.mp4']
audio         -> ['song.mp3']
spreadsheets  -> ['sheet.csv']
docs          -> ['first.txt', 'second.txt', 'third.txt']
pdfs          -> ['report.pdf']
code          -> ['script.py']
others        -> ['archive.zip']
✓ all category filters correct


## 8 · Exact `contentType` filter (still supported)

When `category` is omitted, `contentType` performs an exact MIME match.

In [12]:
body = list_files(FOLDER, contentType='application/pdf')
entries = files_only(body)
print('contentType=application/pdf ->', names(entries))
assert names(entries) == ['report.pdf'], f'exact filter wrong: {names(entries)}'
assert all(e.get('ContentType') == 'application/pdf' for e in entries)
print('✓ exact content-type filter correct')

contentType=application/pdf -> ['report.pdf']
✓ exact content-type filter correct


## 9 · Combined filter + sort

Filter to `images` and sort by `size` descending: `pic-a.png`(5000) then `pic-b.png`(3000).

In [13]:
body = list_files(FOLDER, sortBy='size', sortOrder='desc', category='images')
entries = files_only(body)
print('images, size desc:', list(zip(names(entries), sizes(entries))))
assert names(entries) == ['pic-a.png', 'pic-b.png'], f'combined sort+filter wrong: {names(entries)}'
assert sizes(entries) == sorted(sizes(entries), reverse=True)
print('✓ combined filter + sort correct')

images, size desc: [('pic-a.png', 0.004772186279296875), ('pic-b.png', 0.002864837646484375)]
✓ combined filter + sort correct


## 10 · Invalid category is rejected

An unknown `category` returns a `400 Bad Request`.

In [14]:
resp = requests.post(f'{BASE}/storage/files',
    headers={'Authorization': f'Bearer {token}'},
    json={'path': FOLDER, 'driveId': drive_id, 'category': 'not-a-category'})
print(f'invalid category status : {resp.status_code}  (expect 400)')
assert resp.status_code == 400, f'expected 400 for bad category, got {resp.status_code}'
print('✓ invalid category rejected')

invalid category status : 400  (expect 400)
✓ invalid category rejected


## 11 · Cleanup

Delete the fixture folder and its contents.

In [15]:
# Clean slate: drop any folder left over from a previous run (ignoring the
# error — it usually doesn't exist). Otherwise a stale folder's metadata rows
# make each upload take the S3 overwrite path, which tries to CopyObject an
# object that no longer exists and fails with a 400.
requests.post(f'{BASE}/storage/folder/delete',
    headers={'Authorization': f'Bearer {token}'},
    json={'path': FOLDER, 'driveId': drive_id})

resp = requests.post(f'{BASE}/storage/folder/create',
    headers={'Authorization': f'Bearer {token}'},
    json={'path': FOLDER, 'driveId': drive_id})
assert resp.status_code == 200, f'create folder failed: {resp.status_code} {resp.text}'
print(f'✓ folder "{FOLDER}" created')

✓ folder "sort_filter_1787165718" created
